In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   Signature Detection: YOLOv5nu · YOLOv8n · Ensemble (WBF)     ║
# ║   Google Colab Notebook — run cells top to bottom               ║
# ╚══════════════════════════════════════════════════════════════════╝


# ══════════════════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ══════════════════════════════════════════════════════════════════

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("ultralytics", "pandas", "matplotlib", "opencv-python-headless")
print("✔ All packages installed.")


# ══════════════════════════════════════════════════════════════════
# CELL 2 — Download & prepare dataset
# ══════════════════════════════════════════════════════════════════

import os, zipfile, shutil, yaml, urllib.request
from pathlib import Path

DATASET_URL = (
    "https://github.com/ultralytics/assets/releases/"
    "download/v0.0.0/signature.zip"
)
DATASET_ZIP = Path("signature.zip")
DATASET_DIR = Path("signature_data")
OUTPUT_DIR  = Path("sig_runs")
OUTPUT_DIR.mkdir(exist_ok=True)


def download_dataset(url, dest):
    def _progress(block_num, block_size, total_size):
        done = min(block_num * block_size, total_size)
        pct  = done / total_size * 100 if total_size > 0 else 0
        bar  = "█" * int(pct // 2) + "░" * (50 - int(pct // 2))
        print(f"\r  [{bar}] {pct:5.1f}%  ({done/1e6:.1f}/{total_size/1e6:.1f} MB)",
              end="", flush=True)
    print(f"Downloading dataset …\n  {url}\n")
    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print(f"\n✔ Saved → {dest}  ({dest.stat().st_size/1e6:.1f} MB)")


def prepare_dataset():
    if (DATASET_DIR / "images" / "train").exists():
        n_tr = len(list((DATASET_DIR / "images" / "train").glob("*.jpg")))
        n_vl = len(list((DATASET_DIR / "images" / "val").glob("*.jpg")))
        print(f"Dataset already present: {n_tr} train / {n_vl} val images.")
        return DATASET_DIR
    if not DATASET_ZIP.exists():
        download_dataset(DATASET_URL, DATASET_ZIP)
    print("Extracting …")
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(DATASET_DIR)
    # Flatten nested sub-folder if present
    subdirs = [d for d in DATASET_DIR.iterdir() if d.is_dir()]
    if len(subdirs) == 1 and not (DATASET_DIR / "images").exists():
        inner = subdirs[0]
        for item in inner.iterdir():
            shutil.move(str(item), str(DATASET_DIR / item.name))
        inner.rmdir()
    n_tr = len(list((DATASET_DIR / "images" / "train").glob("*.jpg")))
    n_vl = len(list((DATASET_DIR / "images" / "val").glob("*.jpg")))
    print(f"✔ Extracted: {n_tr} train / {n_vl} val images.")
    return DATASET_DIR


data_root = prepare_dataset()

yaml_path = OUTPUT_DIR / "signature.yaml"
with open(yaml_path, "w") as fh:
    yaml.dump({
        "path" : str(data_root.resolve()),
        "train": "images/train",
        "val"  : "images/val",
        "names": {0: "signature"},
    }, fh, default_flow_style=False)

print(f"YAML → {yaml_path}")
print(f"Root → {data_root.resolve()}")


# ══════════════════════════════════════════════════════════════════
# CELL 3 — Shared training config
# ══════════════════════════════════════════════════════════════════

import torch
from ultralytics import YOLO

EPOCHS     = 50
IMG_SIZE   = 640
BATCH      = 16
WORKERS    = 2
CONF_THRES = 0.35
IOU_THRES  = 0.45
DEVICE     = "0" if torch.cuda.is_available() else "cpu"
RUN_NAME_V5 = "yolov5n_sig"
RUN_NAME_V8 = "yolov8n_sig"

print(f"Device  : {'GPU ✔' if DEVICE == '0' else 'CPU (slower)'}")
print(f"Epochs  : {EPOCHS}  |  Img: {IMG_SIZE}  |  Batch: {BATCH}")

SHARED = dict(
    data          = str(yaml_path),
    epochs        = EPOCHS,
    imgsz         = IMG_SIZE,
    batch         = BATCH,
    workers       = WORKERS,
    device        = DEVICE,
    project       = str(OUTPUT_DIR),
    conf          = CONF_THRES,
    iou           = IOU_THRES,
    optimizer     = "AdamW",
    lr0           = 0.001,
    lrf           = 0.01,
    warmup_epochs = 3,
    augment       = True,
    exist_ok      = True,
    verbose       = False,
    plots         = True,
)


def find_best_weights(run_name: str) -> Path:
    """
    Ultralytics may save runs to either:
      - <project>/<name>/weights/best.pt          (local / script mode)
      - /content/runs/detect/<project>/<name>/... (Colab default)
    This function checks both and returns whichever exists.
    """
    candidates = [
        OUTPUT_DIR / run_name / "weights" / "best.pt",
        Path("/content/runs/detect") / OUTPUT_DIR.name / run_name / "weights" / "best.pt",
        Path("/content/runs/detect") / run_name / "weights" / "best.pt",
    ]
    for p in candidates:
        if p.exists():
            print(f"  Found weights: {p}")
            return p

    # Last resort: glob search under /content
    matches = list(Path("/content").glob(f"**/{run_name}/weights/best.pt"))
    if matches:
        print(f"  Found weights (glob): {matches[0]}")
        return matches[0]

    raise FileNotFoundError(
        f"Could not find best.pt for run '{run_name}'.\n"
        f"Searched: {[str(c) for c in candidates]}"
    )


# ══════════════════════════════════════════════════════════════════
# CELL 4 — Train YOLOv5nu
# ══════════════════════════════════════════════════════════════════

# yolov5nu.pt — Ultralytics YOLOv5 nano (COCO pretrained)
# Backbone : CSP-Darknet + C3 modules
# Head     : Anchor-based, 3 scales × 3 anchors

print("=" * 58)
print("  TRAINING  YOLOv5nu")
print("=" * 58)

model_v5 = YOLO("yolov5nu.pt")
results_v5 = model_v5.train(**SHARED, name=RUN_NAME_V5)

v5_weights = find_best_weights(RUN_NAME_V5)
print(f"\n✔ YOLOv5nu best weights → {v5_weights}")


# ══════════════════════════════════════════════════════════════════
# CELL 5 — Train YOLOv8n
# ══════════════════════════════════════════════════════════════════

# yolov8n.pt — Ultralytics YOLOv8 nano (COCO pretrained)
# Backbone : C2f modules (improved gradient flow vs CSP)
# Head     : Anchor-free decoupled (separate regression + cls)

print("=" * 58)
print("  TRAINING  YOLOv8n")
print("=" * 58)

model_v8 = YOLO("yolov8n.pt")
results_v8 = model_v8.train(**SHARED, name=RUN_NAME_V8)

v8_weights = find_best_weights(RUN_NAME_V8)
print(f"\n✔ YOLOv8n best weights  → {v8_weights}")


# ══════════════════════════════════════════════════════════════════
# CELL 6 — Evaluate individual models
# ══════════════════════════════════════════════════════════════════

print("=" * 58)
print("  EVALUATION — Individual Models")
print("=" * 58)

best_v5 = YOLO(str(v5_weights))
best_v8 = YOLO(str(v8_weights))

_val = dict(data=str(yaml_path), imgsz=IMG_SIZE,
            conf=CONF_THRES, iou=IOU_THRES, verbose=False)

metrics_v5 = best_v5.val(**_val)
metrics_v8 = best_v8.val(**_val)


def extract_metrics(m) -> dict:
    return {
        "mAP50"    : float(m.box.map50),
        "mAP50-95" : float(m.box.map),
        "Precision": float(m.box.mp),
        "Recall"   : float(m.box.mr),
    }


met_v5 = extract_metrics(metrics_v5)
met_v8 = extract_metrics(metrics_v8)

print(f"\n  YOLOv5nu → mAP50={met_v5['mAP50']:.4f}  "
      f"mAP50-95={met_v5['mAP50-95']:.4f}  "
      f"P={met_v5['Precision']:.4f}  R={met_v5['Recall']:.4f}")
print(f"  YOLOv8n  → mAP50={met_v8['mAP50']:.4f}  "
      f"mAP50-95={met_v8['mAP50-95']:.4f}  "
      f"P={met_v8['Precision']:.4f}  R={met_v8['Recall']:.4f}")


# ══════════════════════════════════════════════════════════════════
# CELL 7 — Ensemble: Weighted Box Fusion (WBF)
# ══════════════════════════════════════════════════════════════════

import numpy as np


def wbf_single_image(boxes_list, scores_list, labels_list,
                     weights=None, iou_thr=0.50, skip_thr=0.001):
    """
    Weighted Box Fusion for one image.

    Unlike NMS (keeps only 1 box, discards others), WBF merges ALL
    overlapping predictions from every model via a weighted spatial
    average, preserving localisation signal from both architectures.

    Args:
        boxes_list  : list of (N_i, 4) arrays [x1,y1,x2,y2] in [0,1]
        scores_list : list of (N_i,) confidence arrays
        labels_list : list of (N_i,) class-id arrays
        weights     : per-model importance (default: equal)
        iou_thr     : IoU threshold to merge a box into a cluster
        skip_thr    : ignore predictions below this confidence
    Returns:
        fused_boxes (M,4), fused_scores (M,), fused_labels (M,)
    """
    if weights is None:
        weights = [1.0] * len(boxes_list)

    # Step 1 — collect all candidates
    cands = []
    for mi, (bbs, scs, lbs) in enumerate(
            zip(boxes_list, scores_list, labels_list)):
        for box, sc, lb in zip(bbs, scs, lbs):
            if sc >= skip_thr:
                cands.append({"ws": float(sc) * weights[mi],
                               "box": [float(v) for v in box],
                               "cls": int(lb)})
    if not cands:
        return np.zeros((0,4)), np.zeros(0), np.zeros(0, dtype=int)

    # Step 2 — sort by weighted score
    cands.sort(key=lambda c: -c["ws"])

    # Step 3 — greedy cluster assignment
    clusters = []
    for cand in cands:
        bx1, by1, bx2, by2 = cand["box"]
        placed = False
        for cl in clusters:
            if cl[0]["cls"] != cand["cls"]:
                continue
            tw  = sum(c["ws"] for c in cl)
            rx1 = sum(c["ws"]*c["box"][0] for c in cl)/tw
            ry1 = sum(c["ws"]*c["box"][1] for c in cl)/tw
            rx2 = sum(c["ws"]*c["box"][2] for c in cl)/tw
            ry2 = sum(c["ws"]*c["box"][3] for c in cl)/tw
            ix1 = max(bx1,rx1); iy1 = max(by1,ry1)
            ix2 = min(bx2,rx2); iy2 = min(by2,ry2)
            inter = max(0.,ix2-ix1)*max(0.,iy2-iy1)
            union = ((bx2-bx1)*(by2-by1)+(rx2-rx1)*(ry2-ry1)-inter+1e-7)
            if inter/union >= iou_thr:
                cl.append(cand); placed = True; break
        if not placed:
            clusters.append([cand])

    # Step 4 — fuse each cluster into one box
    out_boxes, out_scores, out_labels = [], [], []
    for cl in clusters:
        tw = sum(c["ws"] for c in cl)
        out_boxes.append([sum(c["ws"]*c["box"][i] for c in cl)/tw
                          for i in range(4)])
        out_scores.append(min(tw/(len(cl)*max(weights)), 1.0))
        out_labels.append(cl[0]["cls"])

    return (np.array(out_boxes),
            np.array(out_scores),
            np.array(out_labels, dtype=int))


def run_ensemble_on_val(model_a, model_b, val_img_dir,
                        w_a=0.45, w_b=0.55):
    val_imgs = sorted(Path(val_img_dir).glob("*.jpg"))
    lbl_dir  = Path(str(val_img_dir).replace("images", "labels"))
    all_fused, all_gts = [], []

    for img_path in val_imgs:
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        gt_boxes = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) == 5:
                    _, cx, cy, w, h = map(float, parts)
                    gt_boxes.append([cx-w/2, cy-h/2, cx+w/2, cy+h/2])
        all_gts.append(np.array(gt_boxes) if gt_boxes else np.zeros((0,4)))

        pa = model_a.predict(str(img_path), imgsz=IMG_SIZE,
                              conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
        pb = model_b.predict(str(img_path), imgsz=IMG_SIZE,
                              conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
        ih, iw = pa.orig_shape
        norm = np.array([iw, ih, iw, ih], dtype=float)

        def _extract(pred):
            bbs = pred.boxes
            if bbs is None or len(bbs)==0:
                return np.zeros((0,4)), np.zeros(0), np.zeros(0,dtype=int)
            return (bbs.xyxy.cpu().numpy()/norm,
                    bbs.conf.cpu().numpy(),
                    bbs.cls.cpu().numpy().astype(int))

        ba,sa,la = _extract(pa)
        bb,sb,lb = _extract(pb)
        fb,fs,fl = wbf_single_image([ba,bb],[sa,sb],[la,lb],
                                     weights=[w_a,w_b], iou_thr=IOU_THRES)
        all_fused.append({"boxes":fb,"scores":fs,"labels":fl})

    return all_fused, all_gts, val_imgs


def compute_map50(all_dets, all_gts, iou_thresh=0.5):
    tp_l, fp_l, sc_l = [], [], []
    n_gt = 0
    for dets, gts in zip(all_dets, all_gts):
        n_gt += len(gts)
        boxes  = dets["boxes"]; scores = dets["scores"]
        if len(boxes)==0: continue
        order   = np.argsort(-scores)
        boxes   = boxes[order]
        matched = np.zeros(len(gts), dtype=bool)
        for box, sc in zip(boxes, scores[order]):
            sc_l.append(float(sc))
            if len(gts)==0: tp_l.append(0); fp_l.append(1); continue
            ix1=np.maximum(box[0],gts[:,0]); iy1=np.maximum(box[1],gts[:,1])
            ix2=np.minimum(box[2],gts[:,2]); iy2=np.minimum(box[3],gts[:,3])
            inter=np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
            a_p=(box[2]-box[0])*(box[3]-box[1])
            a_g=(gts[:,2]-gts[:,0])*(gts[:,3]-gts[:,1])
            iou=inter/(a_p+a_g-inter+1e-7)
            best=int(np.argmax(iou))
            if iou[best]>=iou_thresh and not matched[best]:
                tp_l.append(1); fp_l.append(0); matched[best]=True
            else:
                tp_l.append(0); fp_l.append(1)
    if not sc_l: return 0.,0.,0.
    order=np.argsort(-np.array(sc_l))
    tp=np.cumsum(np.array(tp_l)[order])
    fp=np.cumsum(np.array(fp_l)[order])
    rec=tp/(n_gt+1e-7); prec=tp/(tp+fp+1e-7)
    ap=sum((prec[rec>=t].max() if np.any(rec>=t) else 0.)
           for t in np.linspace(0,1,11))/11.
    f1=2*prec*rec/(prec+rec+1e-7)
    best=int(np.argmax(f1))
    return float(ap), float(prec[best]), float(rec[best])


# Run ensemble on validation set
val_img_dir = data_root / "images" / "val"
fused_dets, all_gts, val_imgs = run_ensemble_on_val(
    best_v5, best_v8, val_img_dir, w_a=0.45, w_b=0.55)
ap_ens, p_ens, r_ens = compute_map50(fused_dets, all_gts)
met_ens = {"mAP50": ap_ens, "Precision": p_ens, "Recall": r_ens}

print(f"Ensemble WBF → mAP50={ap_ens:.4f}  P={p_ens:.4f}  R={r_ens:.4f}")


# ══════════════════════════════════════════════════════════════════
# CELL 8 — Results summary
# ══════════════════════════════════════════════════════════════════

best_map = max(met_v5["mAP50"], met_v8["mAP50"])

summary = f"""
{'='*64}
  SIGNATURE DETECTION — FINAL RESULTS
{'='*64}
  {'Model':<22} {'mAP50':>7} {'mAP50-95':>10} {'Prec':>8} {'Recall':>8}
  {'-'*60}
  {'YOLOv5nu':<22} {met_v5['mAP50']:>7.4f} {met_v5['mAP50-95']:>10.4f} {met_v5['Precision']:>8.4f} {met_v5['Recall']:>8.4f}
  {'YOLOv8n':<22} {met_v8['mAP50']:>7.4f} {met_v8['mAP50-95']:>10.4f} {met_v8['Precision']:>8.4f} {met_v8['Recall']:>8.4f}
  {'Ensemble (WBF)':<22} {ap_ens:>7.4f} {'—':>10} {p_ens:>8.4f} {r_ens:>8.4f}
  {'-'*60}
  Ensemble improvement over best single model:
    mAP@0.5   : {ap_ens - best_map:+.4f}
    Precision : {p_ens  - max(met_v5['Precision'],met_v8['Precision']):+.4f}
    Recall    : {r_ens  - max(met_v5['Recall'],   met_v8['Recall']):+.4f}

  WHY WBF IMPROVES OVER SINGLE MODELS
  ─────────────────────────────────────
  YOLOv5nu  anchor-based (CSP): strong on clear signatures.
  YOLOv8n   anchor-free (C2f): better on small/edge cases.
  WBF       spatially averages overlapping predictions from
            both models instead of discarding one (as NMS
            would do), reducing localisation error.
{'='*64}
"""
print(summary)
(OUTPUT_DIR / "results_summary.txt").write_text(summary)
print(f"✔ Saved → {OUTPUT_DIR}/results_summary.txt")


# ══════════════════════════════════════════════════════════════════
# CELL 9 — Plot A: Metric comparison bar chart
# ══════════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COL5 = "#1565C0"   # blue
COL8 = "#B71C1C"   # red
COLE = "#2E7D32"   # green

metric_keys = ["mAP50", "Precision", "Recall"]
v5_vals  = [met_v5[k]  for k in metric_keys]
v8_vals  = [met_v8[k]  for k in metric_keys]
ens_vals = [met_ens[k] for k in metric_keys]

x = np.arange(len(metric_keys)); bw = 0.23
fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x-bw, v5_vals,  bw, label="YOLOv5nu",       color=COL5, alpha=0.88)
b2 = ax.bar(x,    v8_vals,  bw, label="YOLOv8n",        color=COL8, alpha=0.88)
b3 = ax.bar(x+bw, ens_vals, bw, label="Ensemble (WBF)", color=COLE, alpha=0.88)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.008, f"{h:.3f}",
                ha="center", va="bottom", fontsize=8.5, fontweight="bold")

for i, (ev, v5v, v8v) in enumerate(zip(ens_vals, v5_vals, v8_vals)):
    gain  = ev - max(v5v, v8v)
    color = COLE if gain >= 0 else "#C62828"
    sym   = "↑" if gain >= 0 else "↓"
    ax.annotate(f"{sym}{abs(gain):.3f}",
                xy=(x[i]+bw, ev), xytext=(x[i]+bw, ev+0.06),
                ha="center", fontsize=9, fontweight="bold", color=color,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.1))

ax.set_xticks(x); ax.set_xticklabels(metric_keys, fontsize=12)
ax.set_ylim(0, 1.22); ax.set_ylabel("Score", fontsize=11)
ax.set_title("YOLOv5nu vs YOLOv8n vs Ensemble (WBF) — Val Metrics",
             fontsize=13, fontweight="bold", pad=14)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR/"metric_comparison.png"), dpi=150)
plt.show()
print(f"✔ Saved → {OUTPUT_DIR}/metric_comparison.png")


# ══════════════════════════════════════════════════════════════════
# CELL 10 — Plot B: Training curves from results.csv
# ══════════════════════════════════════════════════════════════════

import pandas as pd

def find_results_csv(run_name):
    """Search for results.csv in the same way we search for weights."""
    candidates = [
        OUTPUT_DIR / run_name / "results.csv",
        Path("/content/runs/detect") / OUTPUT_DIR.name / run_name / "results.csv",
        Path("/content/runs/detect") / run_name / "results.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    matches = list(Path("/content").glob(f"**/{run_name}/results.csv"))
    return matches[0] if matches else None


def load_csv(run_name):
    p = find_results_csv(run_name)
    if p is None:
        return None
    df = pd.read_csv(p)
    df.columns = df.columns.str.strip()
    return df

def first_col(df, *keywords):
    for col in df.columns:
        if all(k in col.lower() for k in keywords):
            return col
    return None

df5 = load_csv(RUN_NAME_V5)
df8 = load_csv(RUN_NAME_V8)

if df5 is not None and df8 is not None:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("Training Dynamics — YOLOv5nu vs YOLOv8n",
                 fontsize=13, fontweight="bold")

    ax = axes[0]
    c5 = first_col(df5, "box", "loss"); c8 = first_col(df8, "box", "loss")
    if c5: ax.plot(df5[c5], color=COL5, lw=2, label="YOLOv5nu")
    if c8: ax.plot(df8[c8], color=COL8, lw=2, label="YOLOv8n")
    ax.set_title("Box Loss (train)"); ax.set_xlabel("Epoch")
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    c5 = first_col(df5,"map50") or first_col(df5,"map_0.5")
    c8 = first_col(df8,"map50") or first_col(df8,"map_0.5")
    if c5: ax.plot(df5[c5], color=COL5, lw=2, label="YOLOv5nu")
    if c8: ax.plot(df8[c8], color=COL8, lw=2, label="YOLOv8n")
    ax.axhline(ap_ens, color=COLE, lw=1.8, ls="--",
               label=f"Ensemble WBF ({ap_ens:.3f})")
    ax.set_title("mAP@0.5 (val)"); ax.set_xlabel("Epoch")
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[2]
    c5 = first_col(df5,"precision"); c8 = first_col(df8,"precision")
    if c5: ax.plot(df5[c5], color=COL5, lw=2, label="YOLOv5nu")
    if c8: ax.plot(df8[c8], color=COL8, lw=2, label="YOLOv8n")
    ax.set_title("Precision (val)"); ax.set_xlabel("Epoch")
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR/"training_curves.png"), dpi=150)
    plt.show()
    print(f"✔ Saved → {OUTPUT_DIR}/training_curves.png")
else:
    print("results.csv not found — skipping training curves plot.")


# ══════════════════════════════════════════════════════════════════
# CELL 11 — Plot C: Sample predictions (GT | v5 | v8 | Ensemble)
# ══════════════════════════════════════════════════════════════════

import cv2

N_SHOW = min(5, len(val_imgs))
fig, axes = plt.subplots(N_SHOW, 4, figsize=(18, 4.5*N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis]

for col, title in enumerate([
    "Ground Truth  (lime)",
    "YOLOv5nu  (blue)",
    "YOLOv8n   (red)",
    "Ensemble WBF  (green)",
]):
    axes[0, col].set_title(title, fontsize=9, fontweight="bold", pad=7)

fig.suptitle("Signature Detection — Sample Validation Predictions",
             fontsize=13, fontweight="bold", y=1.01)


def draw_boxes(ax, img_rgb, boxes_norm, scores, color):
    ih, iw = img_rgb.shape[:2]
    ax.imshow(img_rgb); ax.axis("off")
    for i, box in enumerate(boxes_norm):
        x1,y1 = box[0]*iw, box[1]*ih
        x2,y2 = box[2]*iw, box[3]*ih
        ax.add_patch(mpatches.Rectangle(
            (x1,y1), x2-x1, y2-y1,
            linewidth=2.5, edgecolor=color, facecolor="none"))
        if scores is not None and i < len(scores):
            ax.text(x1, y1-5, f"{scores[i]:.2f}",
                    color=color, fontsize=8, fontweight="bold",
                    bbox=dict(facecolor="white", alpha=0.6,
                              pad=1, edgecolor="none"))


for row in range(N_SHOW):
    img_path = val_imgs[row]
    img_rgb  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ih, iw   = img_rgb.shape[:2]
    norm     = np.array([iw,ih,iw,ih], dtype=float)

    draw_boxes(axes[row,0], img_rgb, all_gts[row], None, "lime")

    pv5 = best_v5.predict(str(img_path), imgsz=IMG_SIZE,
                           conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
    if pv5.boxes is not None and len(pv5.boxes):
        draw_boxes(axes[row,1], img_rgb,
                   pv5.boxes.xyxy.cpu().numpy()/norm,
                   pv5.boxes.conf.cpu().numpy(), COL5)
    else:
        axes[row,1].imshow(img_rgb); axes[row,1].axis("off")

    pv8 = best_v8.predict(str(img_path), imgsz=IMG_SIZE,
                           conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
    if pv8.boxes is not None and len(pv8.boxes):
        draw_boxes(axes[row,2], img_rgb,
                   pv8.boxes.xyxy.cpu().numpy()/norm,
                   pv8.boxes.conf.cpu().numpy(), COL8)
    else:
        axes[row,2].imshow(img_rgb); axes[row,2].axis("off")

    fd = fused_dets[row]
    draw_boxes(axes[row,3], img_rgb, fd["boxes"], fd["scores"], COLE)

plt.tight_layout()
fig.savefig(str(OUTPUT_DIR/"sample_predictions.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"✔ Saved → {OUTPUT_DIR}/sample_predictions.png")


# ══════════════════════════════════════════════════════════════════
# CELL 12 — Download outputs to your local machine
# ══════════════════════════════════════════════════════════════════

try:
    from google.colab import files as colab_files
    for fname in ["metric_comparison.png", "training_curves.png",
                  "sample_predictions.png", "results_summary.txt"]:
        fpath = OUTPUT_DIR / fname
        if fpath.exists():
            colab_files.download(str(fpath))
            print(f"⬇ Downloading {fname}")
        else:
            print(f"  (skipped {fname} — not found)")
except ImportError:
    print("Not in Colab — files saved at:", OUTPUT_DIR.resolve())

✔ All packages installed.
Dataset already present: 143 train / 35 val images.
YAML → sig_runs/signature.yaml
Root → /content/signature_data
Device  : GPU ✔
Epochs  : 50  |  Img: 640  |  Batch: 16
  TRAINING  YOLOv5nu
Ultralytics 8.4.17 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.35, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=sig_runs/signature.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.45, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=30

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇ Downloading metric_comparison.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

⬇ Downloading sample_predictions.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇ Downloading results_summary.txt
